# 04 - Model Registry Deep Dive

This notebook explores the full capabilities of the Snowflake Model Registry: version management, metrics, tags, descriptions, RBAC, and artifact inspection.

**Key Concepts**:
- Models are schema-level objects with full RBAC
- Each version stores: serialized model, metrics, tags, description, lineage
- Default version management for production routing
- Side-by-side version comparison

**Prerequisites**: Run `02_model_training_xgboost.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

session = get_active_session()

reg = Registry(
    session=session,
    database_name="MLOPS_DEMO_DB",
    schema_name="PIPELINES"
)
print(f"Connected to Model Registry: MLOPS_DEMO_DB.PIPELINES")

In [ ]:
# List all registered models
models = reg.show_models()
print("Registered Models:")
models

In [ ]:
# Get a specific model and list its versions
model = reg.get_model("CHURN_PREDICTION_MODEL")
versions = model.show_versions()
print("Model Versions:")
versions

In [ ]:
# Inspect a specific version
mv = model.version("v1")
print(f"Model: {mv.model_name}")
print(f"Version: {mv.version_name}")
print(f"Description: {mv.description}")
print(f"\nMetrics:")
mv.show_metrics()

In [ ]:
# Set tags for lifecycle management and governance
mv.set_tag("stage", "development")
mv.set_tag("owner", "ml-team")
mv.set_tag("use_case", "customer_churn")
mv.set_tag("framework", "xgboost")

print("Tags set on v1:")
print(mv.get_tag("stage"))
print(mv.get_tag("owner"))
print(mv.get_tag("use_case"))
print(mv.get_tag("framework"))

In [ ]:
# Update description
mv.description = "XGBoost churn prediction model trained on customer behavior features. 8 input features, binary classification."
print(f"Updated description: {mv.description}")

In [ ]:
# Compare versions side-by-side
print("Version Comparison:")
print(f"{'':20} {'v1':>15}")
print("-" * 40)

v1 = model.version("v1")
v1_metrics = v1.show_metrics()
print(f"Metrics: {v1_metrics}")

# If v2_tuned exists from HPO notebook
try:
    v2 = model.version("v2_tuned")
    v2_metrics = v2.show_metrics()
    print(f"\nv2_tuned metrics: {v2_metrics}")
except:
    print("\n(Run notebook 03 to create v2_tuned for comparison)")

In [ ]:
# Set default version (used by model.default for production routing)
model.default = "v1"
print(f"Default version set to: v1")
print(f"\nAccessing default: model.default returns version '{model.default.version_name}'")

In [ ]:
# List available functions on the model version
# These are the callable inference methods
functions = mv.show_functions()
print("Available functions:")
functions

In [ ]:
# RBAC: Models inherit schema-level permissions
# Show grants on the model (requires appropriate role)
session.sql("""
SHOW GRANTS ON MODEL MLOPS_DEMO_DB.PIPELINES.CHURN_PREDICTION_MODEL
""").show()

## What to Show in SnowSight

Navigate to **AI/ML > Model Registry > CHURN_PREDICTION_MODEL**:

| Tab | What to Show |
|-----|-------------|
| Versions | All registered versions with creation timestamps |
| Metrics | Logged performance metrics per version |
| Tags | Custom tags for governance and lifecycle |
| Files | Serialized model artifacts (pickle, metadata) |
| Lineage | Data lineage from source tables to this model |
| Functions | Available inference methods (predict, predict_proba) |

**Key message**: The Model Registry is a first-class Snowflake object with full RBAC, audit trail, and governance -- not a bolt-on metadata store.

---

**Next**: `05_batch_inference.ipynb` - Run batch predictions